# 08 — Electronics, propulsion and thermal integration

The BOM keeps source/provenance/confirmation fields visible. Average rail loads
do not qualify transients. CT/CP propeller screening is an estimate, not measured
thrust, and a nominal voltage pass does not establish low-pack launch margin.
Survey software integration is exported separately; the existing OAK/PX4
platform and new ArduPilot wing remain distinct.


## Hardware and software interfaces

**Inputs:** `electronics.yaml`, `payload/survey_cage.yaml`, `design_point.yaml`
and `integration/mapping_profile.yaml`. **Implementation:** `electronics.py`,
`payload.py`, `propulsion.py`, `power.py` and `integration.py`.

Rail current is modeled from average consumer power divided by rail voltage;
regulator loss propagates back to the pack. Rated-current utilization is a useful
budget comparison but does not include servo stalls, compute start-up, cable loss
or camera transients unless separately declared. Static propeller screening uses
$T=C_T\rho n^2D^4$ and $P=C_P\rho n^3D^5$, with $n$ in revolutions/s and $D$ in
metres. The numerical coefficients and loaded-RPM fraction are assumed. Cooling
uses $\dot m=P_{heat}/(c_p\Delta T)$; aperture sizing is not a pressure-drop or
component-temperature prediction.

Plain-text equations: `thrust = CT * density * rev_per_second^2 * diameter^4`;
`shaft_power = CP * density * rev_per_second^3 * diameter^5`;
`cooling_mass_flow = heat / (specific_heat * temperature_rise)`.

The dedicated fixed-wing build uses a Basler-preferred camera and ArduPilot Plane.
The existing OAK/PX4 multirotor stays a separate supported platform. The exported
mapping handoff describes required capture, timing, navigation and resource
contracts; it does not assert that the adjacent repository already implements
those adapters. The inspection revisions recorded in the handoff bound that claim.

**External review:** resolve BOM confirmation fields, rail protection, isolated
trigger/event timing, sensor and body frames, optical-centre and antenna offsets,
raw GNSS association, storage endurance and thermal soak. Follow source links in
the BOM to verify the chosen revision; no online refresh or procurement is
performed by executing this notebook.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=False)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
from dataclasses import replace
acquisition_cam = replace(cam, max_fps=min(cam.max_fps, s["acquisition_rate_cap_hz"]))
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=140)
    display(Image(filename=str(path)))
    plt.close(fig)


In [ ]:
display(pd.DataFrame(d.electronics.bom_rows()))
display(pd.DataFrame([p.summary() for p in d.payloads.values()]))
rows=[]
for config,payload in d.payloads.items():
    tree=d.electronics.power_tree(D.payload_loads_w(payload))
    for rail,load in tree.items():
        if isinstance(load,dict): rows.append({"config":config,"rail":rail,**load})
rails=pd.DataFrame(rows);display(rails)
fig,ax=plt.subplots(figsize=(10,4))
rails.dropna(subset=["utilisation"]).pivot(index="rail",columns="config",values="utilisation").plot.bar(ax=ax)
ax.axhline(1,color="red",ls="--");ax.set(ylabel="Nominal current / rail capacity",xlabel="Power rail")
save(fig,"08_rail_utilisation.png")
print("Open hardware items:")
for item in d.electronics.open_items(): print("-",item)
for name,p in d.payloads.items():
    if p.compute.get("confirm"): print(f"- {name}: {p.compute['confirm']}")
for name,item in next(iter(d.payloads.values())).cage_members.items():
    if item.get("confirm"):print(f"- {name}: {item['confirm']}")


## Wiring responsibilities

```mermaid
flowchart LR
  Battery --> ESC --> Motor
  Battery --> FC[ArduPilot flight controller]
  FC --> FlightGNSS[Independent flight GNSS]
  Battery --> ServoReg[Servo regulator] --> Elevons
  Battery --> ComputeReg[Compute regulator] --> Companion[Orin or CM5]
  FC --> CameraPower[Camera rail]
  CameraPower --> Camera[Preferred GigE camera]
  Camera -->|Ethernet images| Companion
  Camera -->|Isolated exposure event| SurveyGNSS[F9P survey receiver]
  SurveyGNSS -->|Raw observations and event marks| Companion
  FC <-->|MAVLink telemetry| Companion
  Companion --> NVMe
```

Pin assignments, signal voltages, ground/shield strategy and regulator transient
ratings require the final carrier/receiver/camera choices and harness bench checks.


In [ ]:
screen=D.propulsion_checks(d)
display(pd.DataFrame(screen["checks"]));display(screen["assumptions"])
display(pd.DataFrame([D.thermal_report(d)]))
print(screen["status"])
from uavdesign.integration import build_handoff,validate_handoff
display(validate_handoff(build_handoff()))
